# OnboardingAgent Project - Interview Questions and Answers

This guide is focused only on the **Agentic AI Onboarding Document Assistant** project from the resume.

## 1. Quick Project Pitch

### Q1. Tell me about your OnboardingAgent project.

**Answer:**  
OnboardingAgent is an AI-powered document onboarding assistant built as a FastAPI sidecar for the Xeople onboarding platform. It helps new members upload required documents through a conversational widget, extracts fields like document type, reference number, issue date, and expiry date, shows them to the user for confirmation, and then saves the confirmed data back to the existing ASP.NET backend.

The main stack is Python 3.12, FastAPI, AWS Bedrock, AWS Textract, Pydantic, httpx, and a React/TypeScript widget. The project uses an agentic flow where the LLM can call tools such as document parsing and required-document retrieval, but final save decisions remain deterministic and validated by the backend.

### Q2. What business problem did this project solve?

**Answer:**  
The onboarding team had to manually guide applicants through required document uploads and manually review document details. This was repetitive, slow, and error-prone. The assistant automates document guidance, extracts key document fields, validates required fields, and keeps a human confirmation step before saving.

The impact is reduced manual data entry and review effort, faster onboarding, and fewer mistakes because users confirm extracted values before persistence.

### Q3. What was your role in this project?

**Answer:**  
I designed and developed the backend AI sidecar service and core agentic workflow. My work included FastAPI endpoints, AWS Bedrock integration, document parsing, OCR flow, ASP.NET API integration, session management, guardrails, file validation, extraction fallback logic, and structured logging for debugging and observability.

### Q4. Why did you build it as a sidecar instead of modifying the ASP.NET application?

**Answer:**  
The existing ASP.NET system was already the source of truth for members, required documents, and saved document details. Instead of changing that stable system, we built a Python FastAPI sidecar that integrates one-directionally with ASP.NET APIs.

This reduced risk, allowed us to use modern AI libraries in Python, and kept the legacy backend unchanged. The sidecar fetches document requirements and saves confirmed results through existing ASP.NET endpoints.

## 2. Architecture and Flow

### Q5. Explain the high-level architecture.

**Answer:**  
The browser widget talks to the FastAPI sidecar. FastAPI manages sessions, validates files, calls ASP.NET APIs for document metadata, parses uploaded files, uses AWS Bedrock for reasoning and extraction, and saves confirmed values back to ASP.NET.

The flow is:

1. Widget creates a session.
2. Widget fetches the required document list.
3. User selects a document and required field flags are fetched.
4. User uploads a file.
5. File is validated and parsed.
6. Bedrock extracts fields.
7. User confirms or edits fields.
8. Confirmed values and the file are saved to ASP.NET.

### Q6. What are the main backend modules?

**Answer:**  
The main modules are:

| Module | Responsibility |
|---|---|
| `agent.py` | FastAPI agent endpoints such as session, document-list, upload, confirm, chat |
| `bedrock_service.py` | Agentic loop, Bedrock calls, extraction, retries, circuit breaker |
| `document_parser.py` | PDF, image, DOCX, Excel parsing into LLM-ready content |
| `asp_service.py` | Integration with ASP.NET ASMX and handler APIs |
| `session_manager.py` | In-memory or Redis-backed session lifecycle |
| `tool_router.py` | Executes LLM-requested tools |
| `guardrails.py` | Prompt injection, off-topic detection, PII masking, response limits |
| `history_manager.py` | Token cap, base64 pruning, history compaction |
| `security.py` | Filename sanitisation and file validation |

### Q7. Why did you split session creation and document-list fetching?

**Answer:**  
Earlier, session creation and document-list fetching were coupled, which meant ASP.NET failures could affect session creation and browser network traces were harder to debug. We split them into two API calls:

1. `POST /session` creates the session and returns the greeting.
2. `POST /document-list` fetches required documents from ASP.NET.

This improves reliability, makes the UI more responsive, and gives clearer network-level debugging.

### Q8. Why are required fields fetched separately?

**Answer:**  
Required fields are fetched on demand when the user selects a document. This avoids bloating the document-list response and reduces unnecessary ASP.NET calls. The compact document list only contains rendering information, while field requirements are fetched through `/required-fields`.

The confirm endpoint also re-fetches required field flags before saving to avoid stale validation.

### Q9. What endpoints are involved in the upload-confirm flow?

**Answer:**  
The key endpoints are:

| Endpoint | Purpose |
|---|---|
| `POST /api/v1/agent/session` | Create session and greeting |
| `POST /api/v1/agent/document-list` | Fetch compact document list |
| `POST /api/v1/agent/required-fields` | Fetch field flags for selected document |
| `POST /api/v1/agent/upload` | Validate, parse, and extract fields |
| `POST /api/v1/agent/confirm` | Validate confirmed fields and save to ASP.NET |
| `POST /api/v1/agent/chat` | Handle conversational turns |
| `POST /api/v1/agent/delete-doc` | Delete saved document |
| `POST /api/v1/agent/save-doc` | Save edited approved document |

## 3. Agentic AI and Bedrock

### Q10. Why is this project called agentic AI?

**Answer:**  
It is agentic because the LLM is not only generating text. It can decide when to call tools, such as `parse_document` or `get_required_documents`, receive tool results, and then continue the conversation based on those results.

The backend controls the tool execution and validation. The LLM decides the next action inside a bounded agentic loop, but sensitive operations like saving data are deterministic API calls, not LLM decisions.

### Q11. How does the agentic loop work?

**Answer:**  
For each conversation turn, the service sends the system prompt, conversation history, and tool definitions to Bedrock. If Bedrock returns `end_turn`, we return the final text response. If it returns `tool_use`, the backend executes the requested tool through `ToolRouter`, appends a `tool_result` to history, and calls Bedrock again.

The loop has a maximum iteration limit to prevent infinite tool-calling behavior. In normal upload flow, it usually takes two iterations: one tool call and one final response.

### Q12. What tools can the LLM call?

**Answer:**  
The main tools are:

1. `get_required_documents` - fetches and returns required documents.
2. `parse_document` - parses the uploaded file, extracts fields using Bedrock, validates document identity, and returns structured extraction results.

The tool router validates tool names and handles tool failures by returning an error message to the LLM rather than crashing the conversation loop.

### Q13. Why did you use separate Bedrock calls for conversation and extraction?

**Answer:**  
Conversation and extraction have different requirements. Conversation needs slight flexibility, so it uses the chat model with tools and a small temperature. Extraction must be deterministic and JSON-only, so it uses a separate extraction call with temperature `0.0`.

This separation improves reliability because extraction is optimized for structured output, while the agentic loop is optimized for interaction and tool orchestration.

### Q14. Which Bedrock models are used?

**Answer:**  
The current configuration uses Amazon Nova through AWS Bedrock:

| Use case | Model |
|---|---|
| Chat and agentic loop | `amazon.nova-lite-v1:0` |
| Document field extraction | `amazon.nova-pro-v1:0` |
| Fallback | `amazon.nova-lite-v1:0` |

The design also supports Claude-style message APIs through a wrapper around Bedrock Converse.

### Q15. How do you handle Bedrock failures?

**Answer:**  
The service has retry, fallback, and circuit breaker logic. Transient failures like throttling are retried with exponential backoff and jitter. If the primary model is unavailable, the service switches to a fallback model. If repeated failures occur, the circuit breaker opens and fast-fails for a short recovery window to avoid repeated expensive failing calls.

### Q16. How do you prevent the LLM from hallucinating extracted fields?

**Answer:**  
The extraction prompt explicitly tells the model to return valid JSON only and never invent missing values. If a field is not visible, it must return `null` with low confidence. The system also sends expected document type and valid document list context, then validates extracted document identity after extraction.

Additionally, the final save is not automatic. The user reviews and confirms extracted values before they are saved.

### Q17. How do you manage LLM context since Bedrock has no memory?

**Answer:**  
The application stores conversation history in the session and sends the full relevant history with every Bedrock call. Because LLMs are stateless between API calls, we use `HistoryManager` to enforce token caps, prune large base64 image data after confirmation, and compact old confirmed-document turns into summaries.

### Q18. What was the second-upload 502 issue and how did you fix it?

**Answer:**  
The first upload could add large base64 image blocks to conversation history. On the second upload, this accumulated history could exceed Bedrock's context limit and cause a 502 or request-too-large error.

The fix was to prune base64 data after every successful confirmation using `history_manager.prune_base64()` and compact confirmed document turns with `compact_confirmed_document()`. This preserves conversation meaning but removes heavy payloads.

## 4. Document Parsing and OCR

### Q19. What file formats does the system support?

**Answer:**  
It supports PDF, PNG, JPG/JPEG, DOC, DOCX, XLS, and XLSX. PDFs can be digital or scanned. Images use Textract OCR and Claude Vision. DOCX files use python-docx and markitdown. Excel files use openpyxl or xlrd.

### Q20. How do you parse PDFs?

**Answer:**  
For digital PDFs, the system first uses `pdfplumber` to extract the text layer. If the extracted text is too short, it treats the PDF as scanned and renders pages as images using `pypdfium2`. Those image blocks are then sent to the vision-capable extraction flow.

### Q21. How do you parse image documents?

**Answer:**  
For images, the system uses AWS Textract to extract OCR text. It also sends the image content to the vision model so extraction can still work even if OCR is incomplete. Large images are downscaled with Pillow to reduce payload size while preserving readability.

### Q22. Why use both OCR and Vision?

**Answer:**  
OCR is good for explicit text extraction and gives us searchable text, but it can fail on poor-quality scans or unusual layouts. Vision models can reason over the visual layout and extract fields even when OCR is incomplete. Combining both improves accuracy and robustness.

### Q23. What fields are extracted?

**Answer:**  
The main extracted fields are document category, document type, reference number, issue date, and expiry date. Each field includes a value, confidence score, confidence level, and review flag.

### Q24. What happens if extraction returns null fields?

**Answer:**  
There is a fallback recovery path. Sometimes the LLM may return extracted details in free text instead of the expected tool result. If all structured fields are null, the upload endpoint uses regex fallback patterns to recover fields from the agent message. Recovered fields get lower confidence, so the user is asked to verify them.

### Q25. How do you handle date formats?

**Answer:**  
The extraction prompt asks for dates in `DD/MM/YYYY` format. The fallback recovery logic also normalizes dates like `DD-MM-YYYY` into `DD/MM/YYYY` before sending them to the save flow.

## 5. Document Validation and Human-in-the-Loop

### Q26. How do you detect if the user uploaded the wrong document?

**Answer:**  
After extraction, the system compares the detected document type with the expected document type from the selected document. It uses normalization, synonym matching, substring matching, and word-overlap matching.

If the detected type does not match and confidence is above the configured threshold, the system marks it as a wrong document and asks the user to re-upload or verify.

### Q27. Why is wrong-document rejection confidence-gated?

**Answer:**  
Because OCR and LLM extraction may be uncertain. If confidence is low, rejecting the document could create a false negative and frustrate the user. So high-confidence mismatches are rejected, while low-confidence mismatches are flagged for review but still allow user confirmation.

### Q28. Where is Human-in-the-Loop used?

**Answer:**  
Human-in-the-loop is used after extraction. The system shows extracted values with confidence levels and review flags. The user can accept or edit the fields before confirmation. The confirm endpoint logs which fields were modified by the user, which helps measure extraction quality.

### Q29. Why not save extracted values automatically?

**Answer:**  
Documents contain sensitive and compliance-related information. Automatic saving could persist incorrect data. The user confirmation step ensures that extracted fields are reviewed before they are saved to the downstream ASP.NET system.

### Q30. How do you handle mandatory fields?

**Answer:**  
Mandatory field flags are fetched from ASP.NET using `GetRequiredFields`. Before saving, the confirm endpoint re-fetches those flags and validates that required fields like reference number, issue date, or expiry date are present. If anything mandatory is missing, the request is rejected with a validation error.

## 6. ASP.NET Integration

### Q31. How does the FastAPI service integrate with ASP.NET?

**Answer:**  
It uses async `httpx.AsyncClient` to call ASP.NET APIs. The main calls are `GetDocumentList`, `GetRequiredFields`, `SaveDocDetail`, `DocumentHandler.ashx`, `DeleteDoc`, and file download endpoints.

The FastAPI service does not modify ASP.NET. It only consumes and calls existing endpoints.

### Q32. What is special about the ASMX response format?

**Answer:**  
Classic ASMX endpoints return a double-encoded response like:

```json
{"d": "<json_string>"}
```

So the service first parses the outer JSON, then parses the inner `d` string to get the actual document array or result object.

### Q33. How does saving a confirmed document work?

**Answer:**  
Saving uses a two-step ASP.NET flow:

1. Call `SaveDocDetail` through ASMX JSON to create the document detail record and get a `doc_detail_id`.
2. Call `DocumentHandler.ashx` to upload the actual binary file using that returned ID.

Both steps must succeed before the document is marked confirmed in the FastAPI session.

### Q34. What happens if ASP.NET save fails?

**Answer:**  
The document is not marked confirmed. The endpoint returns a friendly retry response with `save_failed=true`. The session is kept active, and the user can try confirming again. This prevents false success when the downstream system did not persist the document.

### Q35. Why do you store temp files by document ID?

**Answer:**  
Temp files are stored as `session._temp_files[document_id]` so the confirm flow always saves the file associated with the specific document being confirmed. This prevents accidental saving of the wrong uploaded file when multiple uploads happen in one session.

## 7. Security

### Q36. What security controls are implemented for file uploads?

**Answer:**  
File uploads go through multiple layers:

1. Filename sanitization.
2. Allowed extension check.
3. File size limit.
4. Magic-byte validation.
5. MIME-to-extension validation.
6. In-memory-only processing.

The system never writes uploaded files to disk.

### Q37. Why is magic-byte validation important?

**Answer:**  
An attacker can rename a malicious file as `.pdf` or `.jpg`. Extension validation alone is not enough. Magic-byte validation checks the actual binary header to confirm the file type matches the claimed extension.

### Q38. How do you protect against prompt injection?

**Answer:**  
Before messages reach Bedrock, the guardrails check common injection patterns like "ignore previous instructions", role-switching, prompt-reveal attempts, and system-tag manipulation. If detected, the request is blocked and logged.

### Q39. How do you handle off-topic questions?

**Answer:**  
The guardrails detect clearly off-topic categories such as weather, jokes, coding tasks, investment, or general company questions. These are answered with a fixed onboarding redirect message without calling Bedrock, which reduces cost and keeps the assistant scoped.

### Q40. How do you handle PII?

**Answer:**  
The system avoids logging PII values. OCR text sent to the LLM is masked for fields like names, addresses, phone numbers, emails, DOB, and MRZ lines while preserving reference numbers and issue/expiry dates needed for extraction.

Logs include metadata like field names, confidence scores, and event names, but not raw extracted values.

### Q41. How is authentication handled?

**Answer:**  
Agent endpoints are protected by JWT authentication. The FastAPI service validates tokens issued by ASP.NET using python-jose. The authenticated member ID comes from JWT claims, not from the request body. In development, there is a bypass flag, but production validation prevents bypass from being enabled.

### Q42. How do you verify session ownership?

**Answer:**  
For session operations, the service checks that the session's member ID matches the authenticated member ID from the JWT. If there is a mismatch, it raises an authentication error. This prevents one member from accessing another member's onboarding session.

## 8. Sessions, Concurrency, and Scalability

### Q43. How is session state managed?

**Answer:**  
The default session manager stores active sessions in memory with indexes by session ID and member ID. It tracks conversation history, required documents, current document state, temp files, last extraction, and other ephemeral upload state.

The design also supports a Redis-backed session manager for multi-instance deployment.

### Q44. Why use session locks?

**Answer:**  
Upload, chat, and confirm can mutate the same session. Without locks, concurrent requests could corrupt session state or save the wrong temp file. Per-session async locks ensure that one mutation happens at a time. If the lock cannot be acquired in time, the service returns a conflict response.

### Q45. How are sessions cleaned up?

**Answer:**  
There is a background cleanup task started during application startup. It periodically removes expired sessions based on inactivity timeout and clears related rate-limit counters.

### Q46. What would you change for horizontal scaling?

**Answer:**  
I would enable Redis for session storage and distributed locks, run multiple FastAPI containers behind Nginx or a load balancer, and use IAM role-based AWS credentials on EC2. The code already abstracts session access behind a protocol, so moving from memory to Redis does not require endpoint changes.

### Q47. Why are files kept in memory instead of disk?

**Answer:**  
The project handles sensitive onboarding documents. Keeping files in memory reduces persistence risk and avoids cleanup issues. Files are only kept during the upload-to-confirm cycle and removed after confirmation.

## 9. Observability and Reliability

### Q48. What observability did you build?

**Answer:**  
The service uses structured JSON logging with request IDs, session IDs, event names, durations, token usage, confidence scores, and pipeline-level events. It also supports CloudWatch metrics and logs. This helps trace a full session from upload to extraction to save.

### Q49. What logs would you check if extraction returned null fields?

**Answer:**  
I would check:

1. `parsing_start` to confirm file type and size.
2. `ocr_complete` to check OCR confidence and text length.
3. `extraction_result_summary` to see null fields and confidence scores.
4. `bedrock_extraction_raw_response` in debug mode if JSON parsing failed.
5. `field_extraction_fallback` to see whether regex recovery was triggered.

### Q50. What logs would you check if save failed?

**Answer:**  
I would check `confirm_field_review` to see submitted field keys, then `asp_save_failed` or `asp_request_failed` for ASP.NET response details. Finally, I would check `savememberdocdetail_failed_document_not_confirmed` to confirm the document was not marked confirmed.

### Q51. How do you reduce Bedrock cost?

**Answer:**  
The service uses grounded deterministic replies for common status questions, off-topic short-circuiting, response length caps, token cap enforcement, base64 pruning, and history compaction. It also separates extraction from chat so extraction is only called during uploads.

### Q52. How do you handle rate limiting?

**Answer:**  
There are limits for session creation per IP, chat messages per session, and uploads per session. When a limit is hit, the service returns a 429 with retry information where applicable. Rejected requests do not consume additional slots.

## 10. Testing and Quality

### Q53. What tests are available for this project?

**Answer:**  
The project uses pytest. Tests cover Bedrock service behavior, file validation, guardrails, history manager, JWT handler, session manager, ASP service behavior, field recovery, and chat scope.

Bedrock calls are mocked, so tests do not need real AWS credentials.

### Q54. How did you test the Bedrock agentic loop?

**Answer:**  
The tests mock `client.messages.create()` using `AsyncMock`. They cover normal text responses, tool-use flow, circuit breaker behavior, fallback model behavior, retries, and extraction JSON parsing.

### Q55. How do you ensure type safety and API consistency?

**Answer:**  
The service uses Pydantic request and response schemas. Endpoints return typed response models rather than raw dictionaries. Internal domain objects use dataclasses, while external API contracts use Pydantic models.

## 11. Design Decisions and Trade-offs

### Q56. Why FastAPI?

**Answer:**  
FastAPI fits this project because it supports async endpoints, typed request and response models, dependency injection, OpenAPI documentation, and high performance. It also works well with async `httpx` calls to ASP.NET and AI service calls.

### Q57. Why use `httpx.AsyncClient` instead of `requests`?

**Answer:**  
The API is async, and external calls to ASP.NET should not block the event loop. `httpx.AsyncClient` supports async HTTP requests, timeouts, retries, and connection reuse. `requests` would block the worker thread.

### Q58. Why use Pydantic settings?

**Answer:**  
Pydantic settings give typed configuration and startup validation. The application validates environment mode, Bedrock timeouts, retry bounds, JWT production safety, and region/model compatibility before serving traffic.

### Q59. Why use confidence scores?

**Answer:**  
Confidence scores help decide what needs review. High-confidence values can be shown as likely correct, while low-confidence values are highlighted for user verification. They also help monitor extraction quality through logs.

### Q60. What was one important technical challenge?

**Answer:**  
One important challenge was managing LLM context across multiple document uploads. Uploaded images can create large base64 blocks, and if those remain in conversation history, later Bedrock calls can exceed context limits. I solved this with base64 pruning after confirmation, history compaction, and token cap enforcement before Bedrock calls.

### Q61. What was another important challenge?

**Answer:**  
ASP.NET integration was challenging because some endpoints used ASMX double-encoded JSON and saving required a two-step sequence: first creating document metadata, then uploading the binary file through `.ashx`. I handled this with a dedicated async service layer, retry logic, clear logging, and confirmation only after both steps succeed.

### Q62. What would you improve next?

**Answer:**  
I would move session storage to Redis for horizontal scaling, add more automated evaluation for extraction accuracy, improve document-specific extraction prompts, and add dashboard-level metrics for field correction rates, OCR quality, and Bedrock latency.

## 12. Resume-Based Deep-Dive Questions

### Q63. Your resume says "Agentic AI". How exactly did you implement agentic orchestration here?

**Answer:**  
I implemented an agentic loop around Bedrock. The model receives tools and can return `tool_use`. The backend executes the tool, appends a `tool_result`, and calls the model again until it returns `end_turn`. This lets the LLM decide when to parse a document, while the backend keeps tool execution, validation, and persistence deterministic.

### Q64. Your resume says "Human-in-the-Loop workflows". Where is HIL in this project?

**Answer:**  
The HIL step is the confirmation screen. The AI extracts document fields, but the user must verify and confirm them before saving. The system also tracks whether users changed fields, which gives a feedback signal about extraction quality.

### Q65. Your resume says "AWS Bedrock and Textract". How did you use them together?

**Answer:**  
Textract extracts OCR text from images. Bedrock receives the OCR text and image content to extract structured fields. Bedrock is also used for the conversational agentic loop. Textract improves raw text extraction, while Bedrock handles reasoning, field mapping, and document understanding.

### Q66. Your resume says "FastAPI microservices". How is this service designed as a microservice?

**Answer:**  
It is a separate FastAPI sidecar with its own API contracts, configuration, logging, error handling, and deployment path. It integrates with the existing ASP.NET backend through HTTP APIs and can be deployed independently using Docker and EC2.

### Q67. Your resume says "Pydantic". How is Pydantic used?

**Answer:**  
Pydantic is used for API request and response schemas and for application settings validation. It ensures structured API contracts, clear validation errors, and typed configuration at startup.

### Q68. Your resume says "observability". What observability exists here?

**Answer:**  
The project has structured JSON logs with request ID, session ID, event names, timings, token usage, confidence scores, and pipeline completion events. It supports CloudWatch logs and metrics. This allows debugging issues like OCR quality, Bedrock latency, ASP.NET save failures, and rate-limit hits.

### Q69. Your resume says "deterministic post-processing". What is an example?

**Answer:**  
Examples include required-field validation before save, date normalization, wrong-document fuzzy matching, regex fallback recovery, confidence-level mapping, and deterministic ASP.NET payload construction. These are code-driven checks, not left to the LLM.

### Q70. Your resume says "enterprise-grade". What makes this production-oriented?

**Answer:**  
It has authentication, session ownership checks, rate limiting, file validation, no disk persistence for sensitive files, structured logging, exception handling, retry and fallback logic, circuit breaker, token management, typed schemas, and a clean separation between AI reasoning and deterministic persistence.

## 13. Scenario-Based Interview Questions

### Q71. A user uploads a passport but selects driver's licence. What happens?

**Answer:**  
The extraction step detects the document type as passport. The document validator compares it with the expected driver's licence type. If the mismatch confidence is high enough, the upload response marks it as a wrong document and asks the user to upload the correct file or verify the details.

### Q72. OCR fails but the image is still readable. Can extraction work?

**Answer:**  
Yes. For image-based documents, the system sends both OCR text and image content. If Textract fails or returns little text, the vision model can still extract fields from the image.

### Q73. ASP.NET is down during document-list fetch. What happens?

**Answer:**  
The session remains valid because session creation is decoupled from document-list fetch. The document-list endpoint returns an error, and the widget can retry fetching documents without recreating the session.

### Q74. ASP.NET is down during confirm. What happens?

**Answer:**  
The confirm endpoint returns a retry response with `save_failed=true`. It does not mark the document as confirmed, and it keeps the session state so the user can retry later.

### Q75. The user opens two tabs and confirms/upload at the same time. How do you handle it?

**Answer:**  
Per-session locks serialize session mutation. Only one request can mutate a session at a time. If another request cannot acquire the lock within the timeout, it gets a conflict response.

### Q76. Bedrock starts throttling. What happens?

**Answer:**  
The service retries throttled calls with exponential backoff and jitter. If retries fail, it can use the fallback model. After repeated failures, the circuit breaker opens and fast-fails temporarily to protect the system.

### Q77. The LLM does not call the `parse_document` tool and returns text instead. What happens?

**Answer:**  
The upload endpoint detects that structured extraction is missing or all fields are null. It then runs fallback regex recovery on the agent's text response and returns recovered fields with low confidence for user verification.

### Q78. A malicious user uploads an executable renamed as PDF. What happens?

**Answer:**  
The extension check may pass, but magic-byte validation fails because the binary header does not match a PDF. The file is rejected before parsing or LLM processing.

### Q79. What happens after the final required document is confirmed?

**Answer:**  
The session marks that document confirmed, checks whether all required documents are complete, prunes and compacts history, clears temp state, and returns a completion message to the widget.

### Q80. What if a required field becomes mandatory after the user selected the document?

**Answer:**  
The confirm endpoint re-fetches field requirements from ASP.NET immediately before save. If the new required field is missing, confirmation is rejected. This prevents stale cached requirements from allowing invalid saves.

## 14. Strong Closing Answer

### Q81. Why is this project a good example of your AI/ML engineering experience?

**Answer:**  
This project combines practical AI engineering with production backend design. It is not only a prompt demo. It includes an agentic Bedrock loop, OCR and document parsing, structured extraction, human confirmation, ASP.NET integration, security controls, rate limiting, retries, circuit breaker, observability, and deployment readiness.

It shows that I can build AI systems that are reliable, secure, integrated with enterprise systems, and usable in real business workflows.

